# Can the subcomponents copy a finished fine-tune, and how sure is the check that says so?

*E34, `experiments/teacher_training`, 2026-10-23 to 2026-10-28. Record: `reports/analyses/2026-10-23_analysis_teacher-training.md`,
with one section per amendment to Amendment 5; Amendment 6 in `reports/analyses/2026-10-28_analysis_e34-amend6.md`. This notebook gives E34 and Amendments 1 to 4 in brief, then Amendments 6 and 5 in full, newest first.*

## E34 so far

**Teacher training** trains offsets on the alive subcomponents' read and write vectors so that the decomposed model
copies a fine-tuned model's predictions, with Γ and Δ frozen. The loss is the KL from the fine-tuned model's
next-token distribution to the student's, so it needs no task labels. The fine-tune left wholly in Δ, the
*do-nothing* state, is the reference.

- **E34.** It works on all three trigger tasks tried. Trained on the task's own text, every copy kept the behaviour and
  passed the faithfulness check, including where the adjoint split had lost the behaviour. Trained on ordinary Pile text
  alone, every copy also got there by the last checkpoint, including the emoticon task, where we had predicted failure.
- **Amendment 1.** The rare trigger positions in the Pile text did not teach the emoticon copy: with every row holding
  a trigger replaced, the copies still reached a target probability of 0.94 and 0.89. The copy's weight change is not
  the fine-tune's.
- **Amendment 2.** Removing also the rows where the trigger is partly on lowered both Pile-text copies' end state, to
  0.87 and 0.84.
- **Amendment 3.** A rank-8 LoRA is copied as well from the task's text, but only partly from Pile text alone.
- **Amendment 4.** A second batch order moved a copy's end state by at most 0.03, which makes Amendment 2's falls real.
  Under an adversary eight times longer every copy stayed at the unchanged model's level, while five of the six
  fine-tunes did not. It also found that two versions of the adversarial check read one fine-tune six times apart.
  Amendment 5 asks why.

## Amendment 6: does teacher training hold up on plain text, on a rare target, with plain SGD, and for a LoRA copied from plain text?

*2026-10-28. Record: `reports/analyses/2026-10-28_analysis_e34-amend6.md`. Figures from
`experiments/teacher_training/plot_amend6.py`.*

### The answer

Teacher training mostly held up. Its copies change ordinary Pile text at most 1.7 times as much as the fine-tunes they
copy. With plain SGD in place of Adam as its optimiser, it failed the adversarial check on every fine-tune tried. On
the rare target neither copy both kept the behaviour and passed, though both sat far closer to the unchanged model
than the fine-tune did. Copied from plain text alone, a LoRA reached about the bar on two tasks of three.

### What we were trying to test

Teacher training gives the alive subcomponents' read and write vectors trainable offsets, with Γ and Δ frozen. Their
loss is $\mathrm{KL}(\text{teacher} \,\|\, \text{state})$, where the *teacher* is the fine-tuned model. A *task-text
copy* trains on the task's own windows with the trigger weighted up. A *Pile-text copy* trains on 8,192 plain Pile rows
and never needs to see the task. Both use Adam at $10^{-5}$ for 1,000 steps unless said otherwise. All readings are on
E34's H100 card, against the unchanged model $W_0$.

The amendment filled four empty cells of the technique comparison.

1. **Harm on plain text.** E34 read the copies' side effects only on the task's own windows. Here they are read on the
   256 held-out Pile rows the adversary also uses. *Plain-text damage* is $\mathrm{KL}(W_0 \,\|\, \text{state})$, the
   mean over every position. It is 0 for the unchanged model. For the fine-tune left in Δ it is the fine-tune's own
   change, the reference. We read copies of three fine-tunes of the word task r1: one by SGD, one by Adam, and a rank-8
   LoRA.
2. **Plain SGD as the optimiser.** E34 found that teacher training with plain SGD on the subcomponents fails late on
   the SGD fine-tune. That could belong to the optimiser or to that one teacher. The same plain-SGD copy was run on the
   Adam fine-tune and on the LoRA. Failing on both points at the optimiser.
3. **A rare target.** On r2 the word " in" must be followed by " Fig", which the model almost never predicts there.
   Component training never passed this task cleanly. We fine-tuned it with SGD at 0.1 and made both copies.
4. **A LoRA from plain text, for longer.** Amendment 3's Pile-text copies of the LoRA stopped at step 1,000 between
   0.52 and 0.88, still rising. Here they ran to 5,000 steps, and at a second batch order to 2,500. This asks whether
   plain text carries a LoRA given time.

**The adversarial check.** An adversary searches the 256 Pile rows for the masks on the subcomponents
that most raise the KL from the state's predictions to the masked model's. The *ratio* divides that KL by the unchanged
model's, so 1 means no easier to break than $W_0$. From $W_0$'s seed floor, a ratio at or below 1.40 passes and one at
or above 1.61 fails. *Keeping the behaviour* means a held-out probability of the target at the trigger of at least 0.9.
For the rare target it means within 0.05 of the fine-tune's own.

**Controls, run first and alone.** The unchanged model had to read its known floor and zero plain-text damage. The
regenerated r1 SGD fine-tune had to reproduce E34. Its plain-SGD copy had to fail as in E34 before the two new
plain-SGD copies could count.

### What we expected, and why

- **Harmless on plain text.** Each copy's plain-text damage at most twice its fine-tune's. High confidence for the
  four full-weight copies, which read 0.55 to 0.97 times their fine-tune on the task windows. Low for the LoRA's
  task-text copy, which read 2.1 times there.
- **The rare target.** Both copies keep the behaviour and pass, at medium-low confidence. Every task-text copy in E34
  had, but component training never managed this task.
- **Plain SGD.** It fails on both new fine-tunes, at medium confidence, because plain SGD on the subcomponents also
  failed late in component training.
- **A LoRA from plain text.** The emoticon and r1 copies reach 0.9 and pass by step 5,000. No guess on r0, whose copy
  had gained only 0.06 between steps 100 and 1,000.
- **A second batch order** lands within 0.05 in target probability and 25% in the ratio on all three tasks, as the
  full-weight copies did in Amendment 4.

### Do the copies harm plain text more than their fine-tune does?

![amend6_plain](../experiments/teacher_training/figures/amend6_plain.png)

**Reading it.** No. Every scored copy changes plain Pile text by at most 1.7 times as much as its fine-tune does,
inside the filed bar of twice. The figure divides each state's plain-text damage by its fine-tune's own. The black line
at 1 is the fine-tune left in Δ, and 0 is the unchanged model. The Pile-text copies sit at the fine-tune's level.
The task-text copies of the SGD and Adam fine-tunes sit below it, at 0.85 and 0.42. The LoRA's task-text copy is the
one above, at 1.74, much as it read 2.1 on the task's windows. The LoRA's plain-SGD copy, not scored, reads 2.6. The
scaled adjoint split damages least because it drops much of the update and, with it, the behaviour. The caveat: plain-text
damage has no batch-order spread yet, which is why the bar is a factor of two.

### Does plain SGD fail whatever the teacher?

![amend6_sgd](../experiments/teacher_training/figures/amend6_sgd.png)

**Reading it.** Yes. With plain SGD at 0.03 in place of Adam, the copy fails the adversarial check on all three r1
fine-tunes at step 1,000. It reads 3.14 on the SGD and Adam fine-tunes and 2.86 on the LoRA, against a fail line of
1.61. The Adam copies of the same three fine-tunes, read on this card in E34 and Amendment 3, sit at the unchanged
model's level. The plain-SGD copies keep the behaviour, at 0.99 or more, so they fail on faithfulness alone. On the
LoRA the plain-SGD copy is worse than doing nothing, since the LoRA left in Δ passes at 1.00. So the late failure
belongs to the optimiser, not to the SGD teacher.

### Can teacher training place the rare target faithfully?

![amend6_rare](../experiments/teacher_training/figures/amend6_rare.png)

**Reading it.** Not quite: by the filed rule neither copy both keeps the behaviour and passes. The fine-tune itself
reached only 0.863 on held-out windows. Left in Δ it reads 11.3 times the unchanged model, a clear fail. The task-text
copy reads 0.808 and a ratio of 1.45. It misses "kept" by 0.006 and the pass line by 0.05, just outside the green
corner on both axes. Both misses are well inside the batch-order spread Amendment 4 measured. At step 100 this copy
had read 0.871, above the fine-tune, and it slid to 0.806 by step 1,000. The Pile-text copy reads 0.739 and 2.48, a
fail. Both copies sit far closer to the unchanged model under the adversary than the fine-tune does. The scaled
adjoint split loses the behaviour, at 0.015. The caveat: one fine-tune, one batch order and one adversary start.

### Does plain text carry a LoRA, given 5,000 steps?

![amend6_lora](../experiments/teacher_training/figures/amend6_lora.png)

**Reading it.** To about the bar on two tasks of three, and the filed rule counts one. The lines are each Pile-text
copy's held-out target probability, averaged over 50 steps. The dots are single held-out readings where the adversary
was run. Every one of those adversarial reads passed, so what is missing is behaviour, not faithfulness. The emoticon
copy was still climbing at step 5,000, where its reading of 0.908 counts as learnt. The r1 copy has hovered at the bar
since about step 1,500. It read 0.902 and passed at step 2,500, but read 0.887 at 5,000, which the filed rule calls a
plateau. The two tasks sit on opposite sides of the bar by checkpoint noise alone. Their 50-step means at step 5,000
are the other way round, 0.883 for emoticon and 0.913 for r1. The r0 copy is still rising, at 0.82.

A second batch order agrees on all three tasks. At step 2,500 it lies within 0.042 in target probability and 11% in
the ratio of the first.

### Does a copy come closer to its teacher on plain text?

![amend6_gap](../experiments/teacher_training/figures/amend6_gap.png)

**Reading it.** For the full-weight fine-tunes, yes; for the LoRA, hardly or not at all. The quantity is the
plain-text distance from teacher to state, $\mathrm{KL}(\text{teacher} \,\|\, \text{state})$. It is divided by the
unchanged model's own distance, $\mathrm{KL}(\text{teacher} \,\|\, W_0)$, so 1 is no closer than $W_0$ and 0 is the
teacher itself. Copies of the three full-weight fine-tunes close two thirds or more of the distance, ending at 0.08 to
0.33. The LoRA's Pile-text copies end at 0.75 on emoticon, 1.12 on r1 and 1.52 on r0, after 5,000 steps on plain text
alone. The LoRA's task-text copy on r1 reads 1.06. On r1 and r0 the copy has moved as far from $W_0$ as the LoRA did,
but not towards the LoRA. Why was not tested.

The plan's decision tree divided the same KL by the teacher's plain-text damage, a KL in the opposite direction. That
reading, the black ticks, gives 0.81, 0.79 and 1.48. Both readings sit far above the tree's one tenth, so they choose
the same branch.

### Did the controls behave?

All passed. The unchanged model read its adversary floor at 0.0748 and a plain-text damage of exactly 0. The regenerated
r1 SGD fine-tune read a target probability of 0.983 and a size of 0.0287, as in E34. Its state left in Δ read 1.588,
as in E34. Its plain-SGD copy failed at 3.14, with a target probability of 0.986, as in E34. Every regenerated LoRA and
Adam fine-tune matched its same-card reference. The Pile-text LoRA copies at step 1,000 reproduced Amendment 3's
readings to four decimals. The r2 fine-tune cleared its gate of 0.5 at 0.863.

### What we learned

Teacher training's harm to plain text is about that of the fine-tune it copies, within a factor of two, for the three
r1 fine-tunes read. So "harmless" now holds on plain text as well as on the task's windows. The map needs Adam: plain
SGD on the subcomponents fails late whatever the teacher, as it did in component training. For the optimiser subgoal
of `GOALS.md`, teacher training works as a map with Adam and not with plain SGD at this rate.

The rare target is still out of reach of every map tried with Γ frozen, though teacher training comes closest. Its
task-text copy misses both bars by less than the known noise, and slid away from the fine-tune as it trained.

Plain text carries a LoRA's trigger behaviour only slowly and to about the bar, and does not carry the LoRA's change
to plain text itself. So the label-free form of the map covers LoRA, the canonical update, only in part.

### How compelling is the evidence

- **One batch order and one adversary start** for everything except the LoRA's Pile-text copies. Each pass is a lower
  bound from one start of 20 steps, as Amendment 5 showed.
- **Plain-text damage is deterministic per state** but has no batch-order spread yet. The factor-of-two bar was set
  wide for that reason.
- **All on r1, except the rare target and the LoRA copies.** The plain-SGD result is one learning rate, 0.03, on three
  teachers of one task.
- **The rare target rests on one fine-tune**, itself below the usual bar at 0.863. The task-text copy's two misses are
  each inside the noise.
- **The LoRA categories rest on a single checkpoint each.** Emoticon's "learnt" and r1's "plateau" swap under a 50-step
  mean.

### Surprises

- **The LoRA copies do not move towards the LoRA on plain text.** Copies of full-weight fine-tunes close two thirds or
  more of the plain-text distance. Copies of the LoRA close a quarter at best, and on r1 and r0 none. Nothing was
  filed on this; the plan had expected the distance to shrink as the trigger behaviour was learnt.
- **The rare-target fine-tune left in Δ reads 11.3 times the unchanged model**, against 1.0 to 1.6 for the r1
  fine-tunes. It is also the largest update read here.
- **The rare target's task-text copy lost behaviour as it trained**, from 0.871 at step 100 to 0.806 at step 1,000.
- **Missed narrowly: r1's LoRA copy read 0.887 at step 5,000**, below the filed 0.9. It had
  read 0.902 at step 2,500.
- **Missed: neither rare-target copy kept and passed.** The task-text copy narrowly, the Pile-text copy clearly.

### What next

1. **The decision tree's approved branch: the r1 and r0 Pile-text LoRA copies to 10,000 steps**, about $1.5 each. If
   r0 reaches 0.9 and passes, plain text carries the LoRA given time. If it stalls below, the label-free map needs
   task text for a LoRA. For r1, which already sits at the bar, a mean over checkpoints would decide more than more
   steps.
2. **The rare target goes to a pending-decision card**, as the tree says for "neither". No map with Γ frozen places
   it, which bears on the "Create" reading and on updating Γ. A cheaper check first: read the task-text copy at step 100
   under the adversary, where it still kept the behaviour.
3. **Why the LoRA's plain-text change is not copied.** Per-matrix norms and cosines of the saved copies against the
   LoRA, on CPU. If the copies' change sits outside the LoRA's matrices, plain text teaches the trigger by a different
   route.

### Scorecard, Amendment 6

Filed 2026-10-28 12:04:51 in `experiments/teacher_training/results/predictions_amend6.json`, sha256 `cab3ba1b…`,
before the run. Scored by the run's own `amend6.score`.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1, W0 | the unchanged model reads its known values | floor about 0.075; plain-text damage at most 1e-4 | 0.0748; 0 | pass |
| G1 | the r1 SGD fine-tune reproduces E34 | P within 0.01, size within 0.003, do-nothing ratio within 0.03 | 0.9834, 0.0287, 1.588 | pass |
| G3 | the plain-SGD copy of it fails as in E34 | "fail", P within 0.02 of 0.9857 | fail 3.14, 0.9857 | pass |
| P1 | copies harmless on plain text | five copies each at most 2× the fine-tune's plain-text damage or 0.002 nats | 0.85, 1.00, 0.42, 0.95, 1.74× | **holds** |
| P2 | r2 task-text copy kept and faithful | P at least the fine-tune's less 0.05; ratio passes | 0.808 against 0.813; 1.45 inconclusive | **fails**: neither |
| P3 | r2 Pile-text copy, two-sided, lean kept and faithful | as P2 | 0.739; 2.48 fail | neither, against the lean |
| P4 | plain SGD as the map, two-sided, lean "map" | fail on both the Adam and LoRA fine-tunes | 3.14 fail, 2.86 fail | **map**, the lean |
| P5 | LoRA from Pile text, 5,000 steps | emoticon and r1 both P ≥ 0.9 and pass | emoticon 0.908 pass; r1 0.887, plateau; r0 0.819, rising | **fails** |
| P6 | a second batch order agrees | each task within 0.05 in P and 25% in ratio at step 2,500 | −0.042/+2%, −0.002/+11%, −0.032/+8% | **holds** |

### Not run, Amendment 6

The r2 Adam fine-tune and a plain-SGD copy on r2. Plain SGD at a second batch order, another learning rate, on emoticon
or r0, or read by the adversary at step 100. The adversary on the copies of stage one, read before in E34 and
Amendments 3 and 4. Pile-text copies past 5,000 steps, on more than 8,192 rows, or at the second batch order past
2,500. A longer adversary or more starts. Any batch-order spread of plain-text damage. The rare-target copies under the
adversary at step 100. Per-matrix analysis of the saved copies. Pyright on `plot_amend6.py`, which could not start on
the worker.

### Provenance, Amendment 6

- RunPod H100 80GB HBM3, run 12:52 to 13:57 on 2026-10-28, 65 minutes, exit 0, about $6.9.
- Results `experiments/teacher_training/results/teacher_training_amend6.json`, sha256 `2b6a7338…`. Log
  `results/large/amend6/run_amend6.log`, `9068de04…`. Saved copies, bf16 offsets of 60 MB each:
  `results/large/amend6/<task>/amend6_*.pt`.
- Code `experiments/teacher_training/amend6.py`, `1d1baed5…`. Figures `figures/amend6_{plain,sgd,rare,lora,gap}.png`
  from `plot_amend6.py`, which also reads E34's and Amendment 3's same-card readings for the plain-SGD figure.

## Amendment 5: why do the two adversaries disagree, and is a copy unique?

*Independently checked 2026-10-26 (`reports/reviews/2026-10-26_review_teacher-training-amend5.md`); corrections applied.*

### The answer

The two adversaries disagree because they start from different random points. Rows, readout precision and step budget
change little. On the r1 SGD fine-tune left in Δ, the budget code reads about 9 times the unchanged model from its own
start. Started where the harness starts, it reads about 1.7, as the harness does. The unchanged model reads nearly the
same from either start. With two starts we cannot say which reading is typical, but a one-start verdict on a fine-tune
can understate the worst case several-fold. Separately, two Pile-text copies with the same target probability have
weight changes that point well apart.

### What we were trying to test

Amendment 4 read one state twice. E34's harness read the r1 SGD teacher's do-nothing state, the fine-tune $\delta W$
left in Δ, at 1.59 times the unchanged model $W_0$. The budget sweep read the same state at 9.24, with the same 20
steps. Every faithfulness verdict in the project rests on one of these two measures, so the gap has to be located
before either is used on the larger model.

**The search both run.** It looks for the masks that make the masked model's predictions differ most from the
state's own. Each subcomponent gets a source $a \in [0, 1]$ and the mask $g + (1 - g)\,a$, with $g$ its causal
importance from Γ. Δ's mask is one more source. One set of sources serves every row. The search starts from random
sources and takes sign steps up the gradient of the mean KL. The reading is the KL in nats per position under the
final masks. The ratio divides it by $W_0$'s reading, so 1 means the state is no easier to break than the unchanged
model.

- **The harness** is the authors' code. It searches two blocks of 128 Pile rows separately and averages them. It
  takes 20 steps of 0.1 and returns its own last objective, computed in bf16.
- **The budget code** is the same search in our code, on the first block only. It takes $n$ steps of $2/n$ and
  reads the KL afterwards in a separate full-precision pass.

Reading the two codes side by side left four differences that could matter: the rows, the readout, the budget and the
start. Both draw their random sources after the same seed. But the harness hands the draws to the matrices in the
model's order, and the budget code in sorted order. So they start from different, equally random numbers. Each reading
below changes one difference, at $W_0$ and at the teacher, adversary seed 0. An identity check runs our code from the
harness's start with the harness's readout. It should give the harness's own number.

**Is a copy unique?** The task-text copy and the Pile-text copy of the same teacher were each retrained at batch
orders 0 and 1. For each pair we read the cosine between the two weight changes over all 24 matrices. A cosine of 1 is
the same direction and 0 is unrelated; a random direction reads about 0.0002.

**Controls, run first.** $W_0$ had to read its known adversary-seed floor. The regenerated teacher had to reproduce
E34's and Amendment 4's readings before any variant was read.

### What we expected, and why

- **The rows do not carry the gap.** Adding the second block, or splitting the harness's blocks, moves the teacher's
  ratio by under 25%, the copies' batch-order spread.
- **One difference in the code path carries it**, the readout or the start. We filed no guess on which; Amendment 4's
  report had suspected the rows or the readout.
- **At $W_0$ every variant reads within 10% of its default**, at high confidence. The two measures had agreed on
  $W_0$ within 0.04 nats.
- **The copies at two batch orders point the same way**, cosine at least 0.8 for both arms, at medium confidence.
  Their target probabilities had moved by under 0.005. Below 0.5 the filed rule reads "one of many": a copy is one
  of many weight changes that behave alike.

### Is the gap in the rows, the readout, the budget, or the start?

![amend5](../experiments/teacher_training/figures/amend5.png)

**Reading it.** On these two draws, the start carries all of it. The figure plots the KL each reading reaches, hollow at the unchanged
model and filled at the teacher, grouped by the start the search took. From the harness's start the teacher reads
about 1.1 to 1.3 nats, or 1.6 to 1.7 times the unchanged model. That holds whichever code runs the search, which rows
it sees, which readout is used and how many steps it takes. From the budget code's start the teacher reads 6 to 7
nats, or 9 to 10 times. The readout moved the teacher's ratio by 0.2%. Adding the second block to one search moved it
by 10%. Eight times as many steps raised each start's reading by about a fifth and left the gap at six times. The
identity check passed: from the harness's start, our code read the harness's number within 4%. The caveat: these are
two random starts on one state at one seed, so we cannot say which reading is typical.

| reading | start | teacher, nats | ratio to $W_0$ |
|---|---|---|---|
| harness, rows 1-128 | harness | 1.084 | 1.60 |
| budget code from the harness's start | harness | 1.128 | 1.67 |
| harness, 160 steps | harness | 1.269 | 1.61 |
| budget code | budget | 6.130 | 9.24 |
| budget code, harness readout | budget | 6.124 | 9.22 |
| budget code, all 256 rows | budget | 7.040 | 10.19 |
| budget code, 160 steps | budget | 7.391 | 10.45 |

### Does the start matter at the unchanged model?

**Reading it.** Hardly at 20 steps, and far less than at the teacher. In the figure above the hollow points sit in a
narrow band. At 20 steps on the first block, $W_0$ reads within 2.4% from either start. At the teacher the same two
starts differ five-fold. Seven of the eight filed comparisons at $W_0$ agree within 4.3%. The one miss is at 160
steps, where the harness's start reads 0.786 nats and the budget code's 0.708. That is 11% apart, against the filed
10%. The two readings differ in the start; their code and readout differences read 4% or less. The budget code's own spread between two adversary seeds at 160 steps
was 23%, so this miss sits inside known noise.

### Does a second batch order give the same copy?

![amend5_cos](../experiments/teacher_training/figures/amend5_cos.png)

**Reading it.** Nearly, for the task-text copy; not for the Pile-text copy. The two task-text copies have a cosine of
0.91, about 24 degrees apart. The two Pile-text copies have a cosine of 0.43, about 65 degrees apart. That is below
the filed 0.5, which the filed rule reads as "one of many". Their target probability is the same, 0.974 at both batch
orders; nothing else about their behaviour was read. Each lines up with the fine-tune's $\delta W$ equally, at about
0.17. A random direction reads at most 0.0002.

The cosine has two blind spots that pull in opposite directions. It cannot see two copies that spread one total change
differently over the subcomponents, which hides differences. It also weighs directions that change no behaviour like
those that carry it. The Pile-text copies train at batch 8 against the task-text copies' 32, so seed-specific drift
with no effect on behaviour could lower their cosine on its own. So 0.43 bounds how alike the total weight changes
are, not where the behaviour sits. One untested fit agrees with such drift: a shared part of about 0.65 of each copy's
length, lining up with $\delta W$ at about 0.25, as the task-text copies' shared part does.

### Did the controls behave?

All passed. The unchanged model read an adversary-seed floor of 0.0748 against 0.075. The regenerated teacher read a
target probability of 0.983 and a size of 0.0287, as in E34. Its harness ratio was 1.588 and its budget ratio 9.24, as
in E34 and Amendment 4. At zero steps the two codes, from the same start, agreed to $4 \times 10^{-8}$.

## What we learned

The disagreement Amendment 4 found is not between two measures but between two random starts of one measure. On this
fine-tune left in Δ, one start finds masks that raise the KL to 6 nats and the other stops near 1. An eight times
longer search did not close the gap. Both are valid searches, so each reading is only a lower bound on the worst
masks. A verdict from one start can therefore be lenient: the harness read this teacher inconclusive at 1.59, where
the other start reads 9.2. The unchanged model barely notices the start, so the sensitivity belongs to the state.

Amendment 4's data already bear on other states: it read all six of its fine-tunes left in Δ from both starts. The
budget code's start read higher on five and lower on none. The r0 SGD fine-tune, which the harness passed at 1.12,
read 2.09, above the budget code's pass line. Those readings also differ in rows, and all six share one pair of draws.
Still, "may be lenient" holds on more than one state, and it reaches a pass. On E37's broad fine-tune the two starts
agreed, and Amendment 4's copies passed from both.

For the larger model: two Pile-text copies with the same target probability have total weight changes at cosine 0.43,
and whether that difference carries behaviour is untested. A claim about where such a copy puts the update needs two
batch orders behind it.

## How compelling is the evidence

- **Two random starts, one adversary seed, one state.** How readings spread over many starts, and on which states, is
  unmeasured.
- **The identity check closes the code path.** At zero steps the codes agreed to $4 \times 10^{-8}$, and after 20
  steps within 4%. No other code difference is hiding.
- **Noise.** The harness's adversary-seed floor at $W_0$ is 0.075. The budget code's two-seed spread is 7% at 20
  steps and 23% at 160. A five- to six-fold gap is far outside both.
- **One teacher and one pair of batch orders for the copies.** E37's copies, trained on their teacher's own rows,
  read 0.93 between batch orders.
- **"Same behaviour" is one number.** Only held-out target probability was read on the copies.

## Surprises

- **Reversed: the Pile-text copies at two batch orders point 65 degrees apart.** We predicted a cosine of at least 0.8
  on both arms, at medium confidence. The Pile-text pair read 0.43, which the filed rule reads as "one of many". Their
  target probabilities agree to 0.0004. Why they differ was not tested.
- **Missed narrowly: at $W_0$, 160 steps from the two starts read 11% apart**, against the filed 10%. Code and readout
  differences read 4% or less.
- **A random start carries a six-fold gap.** Amendment 4's report had suspected the rows or the readout.
- **The task-text copies line up with $\delta W$ at about 0.25**, a little above the plan's expected 0.08 to 0.22.

## What next

1. **For the user's decision: read every adversarial verdict as the maximum over several random starts.** The
   harness's verdicts so far rest on one start. Each further start costs one more search, about 25 s per block on an
   H100. If passing states still pass under the maximum, those passes stand. If not, they are re-read before the
   larger-model run uses the harness.
2. **How many starts.** Read $W_0$ and this teacher from about eight starts, to see whether 1.6 or 9.2 is typical.
3. **Whether the Pile-text copies' difference carries behaviour.** Per-matrix cosines and norms on the saved offsets,
   on CPU, would say whether the copies differ everywhere or in a few matrices.

## Scorecard, Amendment 5

Filed 2026-10-26 20:03:20 in `experiments/teacher_training/results/predictions_amend5.json`, sha256 `f447e43f…`,
before the run. Scored by the run's own `amend5.score`.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1 | the unchanged model reads its known values | floor 0.075 within 5% | 0.0748 | pass |
| G1 | the teacher reproduces E34 and Amendment 4 | P, size, harness and budget ratios | 0.983, 0.0287, 1.588, 9.24 | pass |
| P1 | the rows do not carry the gap | rows256 and harness_b0 within 25% of their defaults | +10%, +0.4% | **holds** |
| P2 | one code-path difference carries it | budget.bf16 or start within 25% of the other's ratio | start 1.67 near the harness's 1.59 points at the start; budget.bf16 9.22 only rules out the readout | **holds** on the start reading |
| P3 | at $W_0$ every variant within 10% | eight pairs | harness160_b0 +11.1% against budget160 | **fails**, narrowly |
| P4 | copies at two batch orders point the same way | cosine ≥ 0.8 on both arms | 0.913 and 0.429 | **fails**: "one of many" |

## Not run

More than two random starts on any state; adversary seeds past 0; other adversary families. The harness at 160 steps
on its second block; our code from the harness's start at 160 steps. Any other teacher under both starts, including
the r0 teachers the harness passed. Harness or budget readings of the four new copies; a third batch order; per-matrix
cosines.

## Provenance

- RunPod H100 80GB HBM3, pod 01M5WYADKPDN6TRMBFYD0KRBV3. Run 20:28 to 20:52, exit 0.
- The pod sat idle until 21:34 while its runner waited at a permission prompt. The lead terminated it after sha256
  checks. Session spend $5.82 for E37 and this amendment together.
- Results `experiments/teacher_training/results/teacher_training_amend5.json`, sha256 `1454bfb4…`. Log
  `results/large/run_amend5.log`, `f18a527f…`. Copies `results/large/r1_t534_v369/amend5_*_step1000.pt`.
- Code `experiments/teacher_training/amend5.py`. Figures `figures/amend5.png` and `figures/amend5_cos.png` from
  `plot_amend5.py`.